### Total Revenue

In [0]:
%sql

SELECT ROUND(SUM(total_revenue_usd), 2) AS total_revenue_usd
FROM novacart_catalog.003_gold.sales_datacube
WHERE order_status = 'completed';

### Revenue by Country

In [0]:
SELECT order_country_code, COUNT(DISTINCT order_id) AS order_count, ROUND(SUM(total_revenue_usd), 2) AS revenue_usd
FROM novacart_catalog.003_gold.sales_datacube
WHERE order_status = 'completed'
GROUP BY order_country_code
ORDER BY revenue_usd DESC;

### Revenue by Channel

In [0]:
SELECT channel, COUNT(DISTINCT order_id) AS order_count, ROUND(SUM(total_revenue_usd), 2) AS revenue_usd
FROM novacart_catalog.003_gold.sales_datacube
WHERE order_status = 'completed'
GROUP BY channel
ORDER BY revenue_usd DESC;

### Completed Order Count

In [0]:
SELECT COUNT(DISTINCT order_id) AS completed_order_count
FROM novacart_catalog.003_gold.sales_datacube
WHERE order_status = 'completed';

### Completed Order Rate

In [0]:
SELECT COUNT(*) AS total_orders,
       COUNT(DISTINCT CASE WHEN order_status = 'completed' THEN order_id END) AS completed_orders,
       ROUND(
         (COUNT(DISTINCT CASE WHEN order_status = 'completed' THEN order_id END) * 100.0) / COUNT(*),
         2
       ) AS completed_order_rate_pct
FROM novacart_catalog.003_gold.sales_datacube;

### Average Order Value (AOV)

In [0]:
SELECT 
  ROUND(
    SUM(total_revenue_usd) / COUNT(DISTINCT order_id),
    2
  ) AS average_order_value_usd
FROM novacart_catalog.003_gold.sales_datacube
WHERE order_status = 'completed';

### Top 5 Products by Revenue

In [0]:
SELECT product_id, product_name, SUM(total_quantity) AS total_quantity_sold,
    ROUND(SUM(total_revenue_usd), 2) AS revenue_usd
FROM novacart_catalog.003_gold.sales_datacube
WHERE order_status = 'completed'
GROUP BY product_id, product_name
ORDER BY revenue_usd DESC
LIMIT 5;

###  Active Customers Count

In [0]:
SELECT 
  COUNT(DISTINCT customer_id) AS active_customer_count
FROM novacart_catalog.003_gold.sales_datacube
WHERE order_status = 'completed';

### Customer Acquisition by Month

In [0]:
SELECT 
  year_month,
  COUNT(DISTINCT customer_id) AS new_customers
FROM novacart_catalog.003_gold.sales_datacube
GROUP BY year_month
ORDER BY year_month;

### Data Quality Score

In [0]:
WITH data_quality_metrics AS (
  SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN product_id IS NOT NULL THEN 1 ELSE 0 END) AS valid_products,
    SUM(CASE WHEN customer_id IS NOT NULL THEN 1 ELSE 0 END) AS valid_customers,
    SUM(CASE WHEN order_id IS NOT NULL THEN 1 ELSE 0 END) AS valid_orders,
    SUM(CASE WHEN total_revenue_usd IS NOT NULL THEN 1 ELSE 0 END) AS valid_revenue
  FROM novacart_catalog.003_gold.sales_datacube
)
SELECT
  ROUND(
    (
      (valid_products * 100.0 / total_records) +
      (valid_customers * 100.0 / total_records) +
      (valid_orders * 100.0 / total_records) +
      (valid_revenue * 100.0 / total_records)
    ) / 4,
    2
  ) AS data_quality_score_pct,
  total_records,
  valid_products,
  valid_customers,
  valid_orders,
  valid_revenue
FROM data_quality_metrics;